# Multiple Linear Regression

*taking the california housing dataset and predicting the given target variable with MLR*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Attempt 1

In [ ]:
# importing tht dataset using sklearn in-build dataset
from sklearn.datasets import fetch_california_housing
fetch_california_housing

In [ ]:
type(fetch_california_housing)

In [ ]:
dataset = fetch_california_housing()
dataset

In [ ]:
# getting fetch error downloading this dataset manually:
df = pd.read_excel("../../../data/fetch_california_housing.xlsx")
df.info()

In [ ]:
df.head()

In [ ]:
df.keys()

In [ ]:
df.describe()

In [ ]:
df.corr()

In [ ]:
sns.heatmap(df.corr(),annot=True)

In [ ]:
sns.pairplot(df.corr())

In [ ]:
cols = df.columns
cols

In [ ]:
num_cols = df.select_dtypes(include="number").columns

for col in num_cols:
    plt.figure(figsize=(5, 2))
    sns.boxplot(x=df[col])
    plt.title(col)
    plt.show()


In [ ]:
target_col = 'target'

# Select only feature columns for cleaning
feature_cols = df.drop(columns=target_col)

# Compute 5-number summary for all features at once
summary = feature_cols.describe().loc[['min', '25%', '50%', '75%', 'max']]
print("5-number summary for features:")
print(summary, "\n")

# Compute IQR for all features
Q1 = feature_cols.quantile(0.25)
Q3 = feature_cols.quantile(0.75)
IQR = Q3 - Q1

# Compute lower and upper fences
lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

# Create mask for all feature columns at once
mask = feature_cols.apply(lambda x: x.between(lower_fence[x.name], upper_fence[x.name]))

# Combine mask across columns: keep rows where all features are within IQR range
mask_all = mask.all(axis=1)

# Apply mask to original dataframe (target stays intact)
df_cleaned = df[mask_all]
# Rows that were removed (outliers)
df_dropped = df[~mask_all]  # ~ inverts the boolean mask

print("Dropped rows (outliers):")
print(df_dropped)

In [ ]:
df_cleaned.shape, df.shape

In [ ]:
df_dropped

In [ ]:
df_cleaned.info()

In [ ]:
# dividing the independent and dependent features
x = df_cleaned.iloc[:, :-1] # independent features
y = df_cleaned.iloc[:, -1] # dependent feature
x

In [ ]:
y

In [ ]:
# splitting the dataset
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(
    x,y,test_size=0.33,random_state=42)

In [ ]:
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
# Standardization
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
# model training:
from sklearn.linear_model import LinearRegression

In [ ]:
lr = LinearRegression()
lr

In [ ]:
lr.fit(x_train_scaled, y_train)

In [ ]:
lr.coef_

In [ ]:
len(cols.drop('target')), len(lr.coef_)

In [ ]:
lr.intercept_

In [ ]:
# prediction of the test data
y_pred = lr.predict(x_test_scaled)
y_pred, y_test

In [ ]:
# performance metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
mse = mean_squared_error(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)
print(f'MSE: {mse}\nMAE: {mae}\nr2_score: {r2}')

In [ ]:
# assumptions
plt.scatter(y_test, y_pred)
plt.xlabel("Test True Data")
plt.ylabel("Test Predicted Data")

In [ ]:
residuals = y_test-y_pred
sns.displot(residuals, kind='kde')

In [ ]:
plt.scatter(y_pred, residuals)

In [ ]:
import pickle as pkl
pkl.dump(lr,open("models/mlr_housing.pkl",'wb'))

In [ ]:
model = pkl.load(open("models/mlr_housing.pkl",'rb'))

In [ ]:
model

In [ ]:
model.predict(x_test_scaled)

In [ ]:
# bad accuracy redoing it!!!

## Redo!!

In [ ]:
df = pd.read_excel("../../../data/fetch_california_housing.xlsx")

In [ ]:
df

In [ ]:
df.info()

In [ ]:
df.corr()

In [ ]:
sns.heatmap(df.corr(),annot = True)

### gotta do:
- **manage the coordinates**
- **handle the high co-linearity**
- **managing the high values using 'Squish' technique (log!)**

In [ ]:
df['Coordinates'] = df['Latitude']*df['Longitude']
df.head()

In [ ]:
cols = df.columns
for col in df.select_dtypes(include="number").columns:
    plt.hist(df[col])
    plt.title(col)
    plt.show()

In [ ]:
import math
num_cols = df.select_dtypes(include="number").columns
n = len(num_cols)

rows = math.ceil(n / 3)   # 3 plots per row
cols = 3

fig, axes = plt.subplots(rows, cols, figsize=(10, 3 * rows))
axes = axes.flatten()  # make indexing easy

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30)
    axes[i].set_title(col)

# Remove empty subplots if any
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# now co-linearity for avgBedroom and avgRooms
df['bedroom_ratio'] = df['AveBedrms']/df["AveRooms"]
df.head()

In [ ]:
# now squishing
df['log_population'] = np.log(df['Population'])
df['log_avg_occupation'] = np.log(df['AveOccup'])
df.head()

In [ ]:
# removing unecessary cols:
df_new = df.drop(["AveBedrms","Population","AveOccup"],axis=1)
df_new.head()

In [ ]:
df_new.corr()

In [ ]:
sns.heatmap(df_new.corr(), annot=True)

In [ ]:
# now dividing dependent and independent features:
x = df_new.drop('target',axis=1)
y=df_new['target']
x

In [ ]:
y

In [ ]:
# now train test split and polynomial features and standardization
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# 1. Prepare your X (ensure Latitude and Longitude are in here!)
X = df_new  # Make sure this has your raw Lat, Lon, and MedInc

# 2. Split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.33, random_state=67)

# 3. Create Polynomial Features (Degree 2 is usually enough)
poly = PolynomialFeatures(degree=2, include_bias=False)
x_train_poly = poly.fit_transform(x_train)
x_test_poly = poly.transform(x_test)

# 4. Standardize the NEW polynomial features
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_poly)
x_test_scaled = scaler.transform(x_test_poly)

# 5. Train your MLR model exactly as before


In [ ]:

# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()

In [ ]:
# x_train_scaled = scaler.fit_transform(x_train)
# x_test_scaled = scaler.transform(x_test)

In [ ]:
# now model training
from sklearn.linear_model import LinearRegression
mlr = LinearRegression()
mlr

In [ ]:
mlr.fit(x_train_scaled, y_train)

In [ ]:
mlr.coef_

In [ ]:
mlr.intercept_

In [ ]:
# prediction of the test data
y_pred_ = mlr.predict(x_test_scaled)
y_pred_, y_test

In [ ]:
# evaluation/ performance metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
mse = mean_squared_error(y_test,y_pred_)
mae = mean_absolute_error(y_test,y_pred_)
r2 = r2_score(y_test,y_pred_)
print(f'MSE: {mse}\nMAE: {mae}\nr2_score: {r2}')

In [ ]:
# assumptions
plt.scatter(y_test, y_pred_)
plt.xlabel("Test True Data")
plt.ylabel("Test Predicted Data")
residuals = y_test-y_pred_

In [ ]:
plt.scatter(y_pred_, residuals)

In [ ]:
import pickle as pkl
pkl.dump(mlr,open("models/mlr_housing2.pkl",'wb'))
model = pkl.load(open("models/mlr_housing2.pkl",'rb'))

In [ ]:
model
model.predict(x_test_scaled)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Create a range of "Fake" Income values from min to max
income_range = np.linspace(x['MedInc'].min(), x['MedInc'].max(), 100)

# 2. Create a "Fake" dataset where only Income changes, others stay at the mean
fake_data = pd.DataFrame([x.mean()] * 100, columns=x.columns)
fake_data['MedInc'] = income_range

# 3. Apply the same transformations (Poly -> Scale)
fake_poly = poly.transform(fake_data)
fake_scaled = scaler.transform(fake_poly)

# 4. Predict prices for this fake data
predicted_curve = mlr.predict(fake_scaled)

# 5. Plot
plt.figure(figsize=(10, 6))
plt.scatter(x_test['MedInc'], y_test, alpha=0.3, label='Actual Data (Test Set)', color='gray')
plt.plot(income_range, predicted_curve, color='red', linewidth=3, label='Polynomial Regression Curve')
plt.xlabel('Median Income')
plt.ylabel('House Price')
plt.title('How the Model "Bends" the Line for Income')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_, alpha=0.3, color='blue')

# Draw the "Perfect Prediction" line
max_val = max(y_test.max(), y_pred_.max())
min_val = min(y_test.min(), y_pred_.min())
plt.plot([min_val, max_val], [min_val, max_val], color='red', lw=2, linestyle='--')

plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted (The Diagonal is the Goal)')
plt.show()